# 2. Éliminatoires aller-retour (4 équipes, 6 journées)

**Problème.** Chaque équipe rencontre deux fois chacune des autres : une fois à domicile, une fois à l'extérieur.
- Les matchs retour commencent quand la phase aller (journées 1 à 3) est terminée.
- Une équipe ne peut pas jouer deux matchs de suite à domicile, ni deux de suite à l'extérieur, sur les paires de journées (1, 2), (3, 4) et (5, 6).

**Variable.** Cette fois, le domicile compte : $x_{ijk} = 1$ si l'équipe $i$ **reçoit** l'équipe $j$ lors de la journée $k$. Donc $x_{ijk}$ et $x_{jik}$ sont deux matchs différents (aller et retour).

**Contraintes.**

$$\sum_{k=1}^{6} x_{ijk} = 1 \quad \forall i \neq j \qquad \text{(C1 : le match « i reçoit j » a lieu une fois)}$$

$$\sum_{j \neq i} \left(x_{ijk} + x_{jik}\right) = 1 \quad \forall i,\ \forall k \qquad \text{(C2 : un match par journée, à domicile ou à l'extérieur)}$$

$$\sum_{k=1}^{3} \left(x_{ijk} + x_{jik}\right) = 1 \quad \forall i < j \qquad \text{(C3 : une rencontre à l'aller ; le retour se déduit de C1 et C3)}$$

$$\sum_{j \neq i} \left(x_{ijk} + x_{ij(k+1)}\right) = 1 \quad \forall i,\ k \in \{1, 3, 5\} \qquad \text{(C4 : exactement un match à domicile par paire, donc ni DD ni EE)}$$

In [1]:
import pulp

equipes = ["A", "B", "C", "D"]
journees = [1, 2, 3, 4, 5, 6]
aller = [1, 2, 3]
paires = [(1, 2), (3, 4), (5, 6)]

## Modèle

In [2]:
modele = pulp.LpProblem("aller_retour", pulp.LpMinimize)

# x[i, j, k] = 1 si i reçoit j lors de la journée k
x = {(i, j, k): pulp.LpVariable(f"x_{i}_{j}_{k}", cat="Binary")
     for i in equipes for j in equipes if i != j for k in journees}

modele += 0

# C1
for i in equipes:
    for j in equipes:
        if i != j:
            modele += pulp.lpSum(x[i, j, k] for k in journees) == 1

# C2
for i in equipes:
    for k in journees:
        modele += pulp.lpSum(x[i, j, k] + x[j, i, k] for j in equipes if j != i) == 1

# C3
for i in equipes:
    for j in equipes:
        if i < j:
            modele += pulp.lpSum(x[i, j, k] + x[j, i, k] for k in aller) == 1

# C4
for i in equipes:
    for (k1, k2) in paires:
        modele += pulp.lpSum(x[i, j, k1] + x[i, j, k2] for j in equipes if j != i) == 1

## Énumération de toutes les solutions

Même principe que pour la phase de groupes : une coupe d'exclusion après chaque calendrier. Un calendrier compte 12 matchs, donc la coupe est $\sum_{(i,j,k) \in S} x_{ijk} \le 11$.

In [3]:
calendriers = []
while True:
    modele.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[modele.status] != "Optimal":
        break
    S = [(i, j, k) for (i, j, k) in x if x[i, j, k].value() > 0.5]
    calendriers.append(S)
    modele += pulp.lpSum(x[v] for v in S) <= len(S) - 1

print(f"Nombre total de calendriers : {len(calendriers)}")

Nombre total de calendriers : 96


In [4]:
# Affichage des 3 premiers calendriers ("B-A" = B reçoit A)
for n, S in enumerate(calendriers[:3], start=1):
    print(f"Calendrier {n}")
    for k in journees:
        print(f"  Journée {k} : " + ", ".join(f"{i}-{j}" for (i, j, kk) in S if kk == k))

Calendrier 1
  Journée 1 : B-A, C-D
  Journée 2 : A-C, D-B
  Journée 3 : A-D, C-B
  Journée 4 : B-C, D-A
  Journée 5 : B-D, C-A
  Journée 6 : A-B, D-C
Calendrier 2
  Journée 1 : B-A, C-D
  Journée 2 : A-C, D-B
  Journée 3 : C-B, D-A
  Journée 4 : A-D, B-C
  Journée 5 : B-D, C-A
  Journée 6 : A-B, D-C
Calendrier 3
  Journée 1 : A-B, D-C
  Journée 2 : B-D, C-A
  Journée 3 : C-B, D-A
  Journée 4 : A-D, B-C
  Journée 5 : A-C, D-B
  Journée 6 : B-A, C-D


## Vérification indépendante

Pour être sûre du nombre obtenu, on le recompte sans solveur : on teste toutes les suites de 6 journées possibles et on garde celles qui respectent les règles.

In [5]:
import itertools

tours = [[("A", "B"), ("C", "D")], [("A", "C"), ("B", "D")], [("A", "D"), ("B", "C")]]

# Une journée = un tour + le choix de qui reçoit dans chaque match
journees_possibles = []
for tour in tours:
    for sens in itertools.product([0, 1], repeat=2):
        journees_possibles.append(tuple(m if s == 0 else m[::-1] for m, s in zip(tour, sens)))

compte = 0
for cal in itertools.product(journees_possibles, repeat=6):
    matchs = [m for jour in cal for m in jour]
    if len(set(matchs)) != 12:                                    # chaque "i reçoit j" une fois
        continue
    if len({frozenset(m) for jour in cal[:3] for m in jour}) != 6:  # toutes les paires à l'aller
        continue
    ok = True
    for e in equipes:
        dom = [any(m[0] == e for m in jour) for jour in cal]
        if any(dom[a] == dom[b] for a, b in [(0, 1), (2, 3), (4, 5)]):
            ok = False
            break
    compte += ok

print("Nombre de calendriers (force brute) :", compte)

Nombre de calendriers (force brute) : 96


## Résultat

Les deux méthodes donnent **96 calendriers**.